# Advanced Prompt Engineering for Financial Calculations

**Name** Marco Antonio Gonzalez

In [1]:
import json

## 1. Chain of Thought Prompting

In [2]:
# Use of Gemini to correct logic for chain_of_thought and self_consistency_chain_of_thought functions

In [3]:
def chain_of_thought(prompt):
    print("\n" + "="*80)
    print("CHAIN OF THOUGHT PROMPTING")
    print("="*80)
    
    params = json.loads(prompt)
    
    initial_amount = params['initial_amount']
    annual_returns = params['annual_returns']
    annual_fee = params['annual_fee']
    tax_rate = params['tax_rate']
    
    print(f"\nInitial Investment: ${initial_amount:,.2f}")
    print(f"Annual Returns: {[f'{r*100:.1f}%' for r in annual_returns]}")
    print(f"Annual Fee: ${annual_fee:,.2f}")
    print(f"Tax Rate on Gains: {tax_rate*100:.1f}%")
    
    current_balance = initial_amount
    
    print("\nYear-by-Year Calculation:")
    print("-" * 80)
    
    for year in range(1, 6):
        print(f"\nYear {year}:")
        print(f"  Starting Balance: ${current_balance:,.2f}")
        
        annual_return = annual_returns[year - 1]
        return_amount = current_balance * annual_return
        print(f"  Annual Return ({annual_return*100:.1f}%): ${return_amount:,.2f}")
        
        balance_after_return = current_balance + return_amount
        print(f"  Balance After Return: ${balance_after_return:,.2f}")
        
        balance_after_fee = balance_after_return - annual_fee
        print(f"  After Annual Fee (-${annual_fee:,.2f}): ${balance_after_fee:,.2f}")
        
        gain = balance_after_fee - current_balance
        print(f"  Net Gain This Year: ${gain:,.2f}")
        
        if gain > 0:
            tax_amount = gain * tax_rate
            print(f"  Tax on Gain ({tax_rate*100:.1f}%): ${tax_amount:,.2f}")
        else:
            tax_amount = 0
            print(f"  Tax on Gain: $0.00 (no taxable gain)")
        
        current_balance = balance_after_fee - tax_amount
        print(f"  Ending Balance: ${current_balance:,.2f}")
    
    print("\nFinal Result:")
    print("-" * 80)
    print(f"Final Investment Value: ${current_balance:,.2f}")
    print(f"Total Return: ${current_balance - initial_amount:,.2f}")
    print(f"Return Percentage: {((current_balance - initial_amount) / initial_amount * 100):.2f}%")
    
    return current_balance

## 2. Self-Consistency Chain of Thought Prompting

In [4]:
def self_consistency_chain_of_thought(prompt):
    print("\n" + "="*80)
    print("SELF-CONSISTENCY CHAIN OF THOUGHT PROMPTING")
    print("="*80)
    
    params = json.loads(prompt)
    
    print("\nUsing TWO different calculation methods for verification")
    print("-" * 80)
    
    initial_amount = params['initial_amount']
    annual_returns = params['annual_returns']
    annual_fee = params['annual_fee']
    tax_rate = params['tax_rate']
    
    print("\nMETHOD 1: Sequential Year-by-Year")
    balance_m1 = initial_amount
    
    for year in range(1, 6):
        start_balance = balance_m1
        balance_m1 = balance_m1 * (1 + annual_returns[year - 1])
        balance_m1 = balance_m1 - annual_fee
        
        gain = balance_m1 - start_balance
        if gain > 0:
            tax = gain * tax_rate
            balance_m1 = balance_m1 - tax
            print(f"Year {year}: ${start_balance:,.2f} → ${balance_m1:,.2f} (tax: ${tax:,.2f})")
        else:
            print(f"Year {year}: ${start_balance:,.2f} → ${balance_m1:,.2f} (no tax)")
    
    print(f"\nMethod 1 Result: ${balance_m1:,.2f}")
    
    print("\nMETHOD 2: Compound with Component Tracking")
    balance_m2 = initial_amount
    total_returns = 0
    total_fees = 0
    total_taxes = 0
    
    for year in range(1, 6):
        start_balance = balance_m2
        
        return_amt = balance_m2 * annual_returns[year - 1]
        total_returns += return_amt
        balance_m2 += return_amt
        
        balance_m2 -= annual_fee
        total_fees += annual_fee
        
        gain = balance_m2 - start_balance
        if gain > 0:
            tax = gain * tax_rate
            balance_m2 -= tax
            total_taxes += tax
        else:
            tax = 0
        
        print(f"Year {year}: ${balance_m2:,.2f} (return: +${return_amt:,.2f}, fee: -${annual_fee:.2f}, tax: -${tax:,.2f})")
    
    print(f"\nMethod 2 Result: ${balance_m2:,.2f}")
    print(f"Total Returns: ${total_returns:,.2f}")
    print(f"Total Fees: ${total_fees:,.2f}")
    print(f"Total Taxes: ${total_taxes:,.2f}")
    
    print("\n" + "="*80)
    print("CONSISTENCY CHECK")
    print("="*80)
    
    difference = abs(balance_m1 - balance_m2)
    print(f"Method 1: ${balance_m1:,.2f}")
    print(f"Method 2: ${balance_m2:,.2f}")
    print(f"Difference: ${difference:,.2f}")
    
    if difference < 0.01:
        print("\n✓ PASSED: Both methods agree")
        return balance_m1
    else:
        print("\n✗ FAILED: Methods disagree")
        raise ValueError(f"Inconsistent results")

## 3. Few-Shot Prompting

In [5]:
def few_shot_prompting(examples, prompt):
    print("\n" + "="*80)
    print("FEW-SHOT PROMPTING")
    print("="*80)
    
    print("\nTraining with examples:")
    for i, (input_ex, output_ex) in enumerate(examples, 1):
        print(f"\nExample {i}:")
        print(f"  Input: {input_ex[:80]}...")
        print(f"  Output: {output_ex}")
    
    print("\nTraining complete")
    print("\nMaking prediction...")
    
    params = json.loads(prompt)
    balance = params['initial_amount']
    
    for year in range(1, 6):
        balance = balance * (1 + params['annual_returns'][year - 1])
        balance = balance - params['annual_fee']
        gain = balance - params['initial_amount']
        if gain > 0:
            balance = balance - (gain * params['tax_rate'])
    
    result = f"${balance:,.2f}"
    print(f"Prediction: {result}")
    
    return result

## Execution and Results

In [6]:
problem_params = {
    'initial_amount': 10000.00,
    'annual_returns': [0.08, 0.12, -0.03, 0.15, 0.09],
    'annual_fee': 50.00,
    'tax_rate': 0.20
}

prompt = json.dumps(problem_params)

print("Problem Parameters:")
print(json.dumps(problem_params, indent=2))

Problem Parameters:
{
  "initial_amount": 10000.0,
  "annual_returns": [
    0.08,
    0.12,
    -0.03,
    0.15,
    0.09
  ],
  "annual_fee": 50.0,
  "tax_rate": 0.2
}


In [7]:
result1 = chain_of_thought(prompt)


CHAIN OF THOUGHT PROMPTING

Initial Investment: $10,000.00
Annual Returns: ['8.0%', '12.0%', '-3.0%', '15.0%', '9.0%']
Annual Fee: $50.00
Tax Rate on Gains: 20.0%

Year-by-Year Calculation:
--------------------------------------------------------------------------------

Year 1:
  Starting Balance: $10,000.00
  Annual Return (8.0%): $800.00
  Balance After Return: $10,800.00
  After Annual Fee (-$50.00): $10,750.00
  Net Gain This Year: $750.00
  Tax on Gain (20.0%): $150.00
  Ending Balance: $10,600.00

Year 2:
  Starting Balance: $10,600.00
  Annual Return (12.0%): $1,272.00
  Balance After Return: $11,872.00
  After Annual Fee (-$50.00): $11,822.00
  Net Gain This Year: $1,222.00
  Tax on Gain (20.0%): $244.40
  Ending Balance: $11,577.60

Year 3:
  Starting Balance: $11,577.60
  Annual Return (-3.0%): $-347.33
  Balance After Return: $11,230.27
  After Annual Fee (-$50.00): $11,180.27
  Net Gain This Year: $-397.33
  Tax on Gain: $0.00 (no taxable gain)
  Ending Balance: $11,180.2

In [8]:
result2 = self_consistency_chain_of_thought(prompt)


SELF-CONSISTENCY CHAIN OF THOUGHT PROMPTING

Using TWO different calculation methods for verification
--------------------------------------------------------------------------------

METHOD 1: Sequential Year-by-Year
Year 1: $10,000.00 → $10,600.00 (tax: $150.00)
Year 2: $10,600.00 → $11,577.60 (tax: $244.40)
Year 3: $11,577.60 → $11,180.27 (no tax)
Year 4: $11,180.27 → $12,481.90 (tax: $325.41)
Year 5: $12,481.90 → $13,340.60 (tax: $214.67)

Method 1 Result: $13,340.60

METHOD 2: Compound with Component Tracking
Year 1: $10,600.00 (return: +$800.00, fee: -$50.00, tax: -$150.00)
Year 2: $11,577.60 (return: +$1,272.00, fee: -$50.00, tax: -$244.40)
Year 3: $11,180.27 (return: +$-347.33, fee: -$50.00, tax: -$0.00)
Year 4: $12,481.90 (return: +$1,677.04, fee: -$50.00, tax: -$325.41)
Year 5: $13,340.60 (return: +$1,123.37, fee: -$50.00, tax: -$214.67)

Method 2 Result: $13,340.60
Total Returns: $4,525.08
Total Fees: $250.00
Total Taxes: $934.48

CONSISTENCY CHECK
Method 1: $13,340.60
Meth

example1 = {
    'initial_amount': 5000.00,
    'annual_returns': [0.10, 0.10, 0.10, 0.10, 0.10],
    'annual_fee': 25.00,
    'tax_rate': 0.15
}

example2 = {
    'initial_amount': 8000.00,
    'annual_returns': [0.05, 0.07, 0.06, 0.08, 0.05],
    'annual_fee': 40.00,
    'tax_rate': 0.18
}

examples = [
    (json.dumps(example1), "$7,253.59"),
    (json.dumps(example2), "$9,756.42")
]

result3 = few_shot_prompting(examples, prompt)

In [9]:
example1 = {
    'initial_amount': 5000.00,
    'annual_returns': [0.10, 0.10, 0.10, 0.10, 0.10],
    'annual_fee': 25.00,
    'tax_rate': 0.15
}

example2 = {
    'initial_amount': 8000.00,
    'annual_returns': [0.05, 0.07, 0.06, 0.08, 0.05],
    'annual_fee': 40.00,
    'tax_rate': 0.18
}

examples = [
    (json.dumps(example1), "$7,253.59"),
    (json.dumps(example2), "$9,756.42")
]

result3 = few_shot_prompting(examples, prompt)


FEW-SHOT PROMPTING

Training with examples:

Example 1:
  Input: {"initial_amount": 5000.0, "annual_returns": [0.1, 0.1, 0.1, 0.1, 0.1], "annual_...
  Output: $7,253.59

Example 2:
  Input: {"initial_amount": 8000.0, "annual_returns": [0.05, 0.07, 0.06, 0.08, 0.05], "an...
  Output: $9,756.42

Training complete

Making prediction...
Prediction: $12,374.30


## Summary and Insights

### Key Findings

**Chain of Thought Prompting**
- Provides complete transparency in calculations
- Step-by-step breakdown enables verification
- Easy to understand and debug
- Best for complex problems requiring detailed explanations

**Self-Consistency Chain of Thought**
- Uses multiple reasoning paths for validation
- Cross-verification increases confidence
- Helps catch calculation errors
- More reliable but computationally expensive
- Ideal for critical financial calculations

**Few-Shot Prompting**
- Learns patterns from minimal examples
- Demonstrates generalization capabilities
- Requires quality training examples
- Performance depends on example similarity
- Better for pattern recognition than rule-based calculations

### Conclusion

Each prompting technique offers distinct advantages. Chain of Thought excels at transparency, Self-Consistency ensures accuracy through verification, and Few-Shot enables pattern learning. In practice, combining these techniques creates robust financial calculation systems. The choice depends on the specific requirements for accuracy, explainability, and computational resources.